# DKI — new model vs **batched legacy** model on keystoneness

This notebook reuses the **structural keystoneness workflow** setup
(`dki_keystoneness_workflow.ipynb`: same clone/install, same data loader, same
leave-one-out machinery and the same `classical_structural_keystoneness` port)
but runs **two models head-to-head on identical data, splits and a shared
training budget**:

* **New model** — the modern `dki` package replicator model with **nonlinear
  SiLU fitness** (`fc2(SiLU(fc1(y)))`), trained via `dki.train.train`.
* **Batched legacy model** — the original cNODE2 dynamics (two stacked
  `Linear(N, N)` layers, *no activation*) in its **batch-safe** form
  (`legacy.baseline_runner.ODEFuncBatched`), integrated one `odeint` call over
  the whole `(B, N)` minibatch. This is the "batch legacy model": same legacy
  fitness, just the batched integration path.

Both share the **same train/val split, batch size, learning rate, epochs and
seed**, so the only thing that varies is the *model* (nonlinear SiLU vs the
legacy collapses-to-linear two-`Linear` map). We then compute structural
keystoneness from each and compare:

1. validation reconstruction (Bray–Curtis) and wall-clock,
2. how strongly the two models' `k_pred` agree,
3. (when removal ground truth exists) which model's `k_pred` better tracks
   `k_true`,
4. their top keystone-species rankings.

> Keystoneness is a **counterfactual from the trained model** (remove a species
> → re-integrate → measure the Bray–Curtis shift), so it is run on **all**
> communities — no leakage concern. The val split's only job is model
> selection; a low `val_bc` is what licenses trusting the keystoneness numbers.

## 1. Setup

In [ ]:
import os
if not os.path.exists('/content/DKI'):
    !git clone https://github.com/metagenAu/DKI.git /content/DKI
%cd /content/DKI
# use the branch this notebook lives on (falls through to whatever is checked out)
!git fetch origin claude/gallant-ride-DXSt0 2>/dev/null && git checkout claude/gallant-ride-DXSt0 2>/dev/null || true
!pip install -q -r requirements.txt
import sys
if '/content/DKI' not in sys.path:
    sys.path.insert(0, '/content/DKI')

In [ ]:
import copy
import time

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from dki.train import TrainConfig, train
from dki.infer import predict
from dki.keystoneness import classical_structural_keystoneness

# legacy building blocks: the batch-safe cNODE2 dynamics + integration helpers
from legacy.baseline_runner import ODEFuncBatched, integrate_set, loss_bc_rows

def spearman(a, b):
    """Spearman rho via pandas (keeps scipy out of the dependency set)."""
    return pd.Series(np.asarray(a)).corr(pd.Series(np.asarray(b)), method='spearman')

print('torch', torch.__version__, '| cuda', torch.cuda.is_available())

## 2. Data
Identical to the keystoneness workflow. `USE_BUNDLED=True` uses the repo's
100-taxa synthetic set, which ships the removal-experiment ground truth
(`Ptest.csv` / `Ztest.csv` / `Sample_id.csv` / `Species_id.csv`), so you also
get the `k_true` column and can ask **which model tracks truth better**. Set it
to `False` to upload your own abundance CSV.

The DKI loader expects `Ptrain.csv` as **(n_taxa, n_samples)** with no header /
index. The flags below transpose / strip headers from your upload to match.
`MIN_READS` is a read-depth filter on the **raw counts**; leave it at `0` for
data already in relative abundance and for the bundled ground-truth set
(filtering reindexes taxa and would desync the on-disk ids).

In [ ]:
USE_BUNDLED = True         # set False to upload your own
samples_as_rows = True     # only matters when USE_BUNDLED=False
header_row = True          # set True if your CSV has a header row
index_col  = True          # set True if your CSV has a row-label column
MIN_READS  = 0             # drop samples with < this many total reads (0 = off)

DATA_DIR = '/content/DKI/data' if USE_BUNDLED else '/content/dki_data'

if not USE_BUNDLED:
    from google.colab import files
    os.makedirs(DATA_DIR, exist_ok=True)
    print('Upload your abundance CSV (and optionally a test CSV).')
    uploaded = files.upload()
    for name in uploaded:
        df = pd.read_csv(name,
                         header=0 if header_row else None,
                         index_col=0 if index_col else None)
        arr = df.to_numpy(dtype=np.float32)
        if samples_as_rows:
            arr = arr.T    # -> (n_taxa, n_samples) for the DKI loader
        is_train = ('train' in name.lower()) or (len(uploaded) == 1)
        dst = os.path.join(DATA_DIR, 'Ptrain.csv' if is_train else 'Ptest.csv')
        np.savetxt(dst, arr, delimiter=',')
        print(f'  wrote {dst}  shape={arr.shape}  (taxa, samples)')

print('Data dir:', DATA_DIR, '| MIN_READS =', MIN_READS)
!ls -la $DATA_DIR

## 3. Shared training budget
To make this a fair head-to-head we hold the **recipe** constant and vary only
the model. Both the new model and the batched legacy model see the same split
(`SEED`, `VAL_FRACTION`), the same `BATCH`, `LR` and `EPOCHS`.

* `LR=1e-3` — the high-dimensional replicator field diverges at `1e-2`, so we
  use the new model's safe rate for **both** to avoid a NaN'd legacy run.
* `BATCH` is capped to the number of training samples available.
* Only the new model uses early stopping; the legacy loop keeps the best-val
  checkpoint (deepcopy) the same way the original did.

In [ ]:
SEED = 0
VAL_FRACTION = 0.2
BATCH = 500          # capped to n_train below
LR = 1e-3
EPOCHS = 300

# Integration grids for the legacy model.
T_TRAIN = torch.arange(0.0, 100.0, 0.01)   # faithful legacy training grid (t in [0,100])
T_PRED = torch.tensor([0.0, 100.0])         # same adaptive solve, fewer stored points -> light memory

## 4. Train the **new** model (nonlinear SiLU fitness)
`train(cfg)` also returns the loaded `DKIData`; we reuse that single `data`
object for **both** models so the splits and tensors are guaranteed identical.

In [ ]:
cfg = TrainConfig(
    data_dir=DATA_DIR, out_dir='/content/results',
    batch_size=BATCH,
    lr=LR, min_lr=1e-5,
    epochs=EPOCHS,
    early_stop_patience=50,
    grad_clip=1.0,
    val_fraction=VAL_FRACTION, seed=SEED,
    min_reads=MIN_READS,
    save_predictions=False,
    nonlinear=True,           # <- the new model: SiLU fitness
)
model, result_new, data = train(cfg)

# Work on CPU copies of the shared tensors so legacy predictions line up taxon-
# for-taxon with the new model's, regardless of which device train() picked.
z_all_cpu = data.z_all.detach().cpu()
N = data.n_species

print(f'New model: {N} taxa, {data.p_all.shape[0]} samples '
      f'(after MIN_READS={MIN_READS}).')
print(f'Best val BC: {result_new.best_val_loss:.6f} at epoch {result_new.best_epoch}')
print(f'Total train time: {np.sum(result_new.epoch_seconds):.0f}s '
      f'({np.mean(result_new.epoch_seconds):.3f}s/epoch)')

## 5. Train the **batched legacy** model (cNODE2, `ODEFuncBatched`)
Same data split / batch / lr / epochs as above, but the model is the original
two-`Linear` fitness with the batch-safe replicator mean-field term, integrated
in a single `odeint` call over the whole minibatch. Best-val checkpoint is kept
via `deepcopy`, exactly as `legacy/baseline_runner.py` does.

In [ ]:
torch.manual_seed(SEED)
np.random.seed(SEED)

ztrn, ptrn_t = data.z_train.detach().cpu(), data.p_train.detach().cpu()
zval, pval = data.z_val.detach().cpu(), data.p_val.detach().cpu()
M = ztrn.shape[0]
bs = min(BATCH, M)

legacy = ODEFuncBatched(N)
opt = torch.optim.Adam(legacy.parameters(), lr=LR)

leg_train_hist, leg_val_hist, leg_times = [], [], []
best_val = float('inf')
best_legacy = copy.deepcopy(legacy)

for e in range(EPOCHS):
    t0 = time.perf_counter()
    s = torch.from_numpy(np.random.choice(np.arange(M, dtype=np.int64), bs, replace=False))
    batch_z, batch_p = ztrn[s], ptrn_t[s]

    opt.zero_grad()
    q_pred = integrate_set(legacy, batch_z, T_TRAIN, batched=True)
    loss = loss_bc_rows(q_pred, batch_p).sum()

    with torch.no_grad():
        q_val = integrate_set(legacy, zval, T_TRAIN, batched=True)
        val_mean = loss_bc_rows(q_val, pval).sum().item() / zval.shape[0]

    leg_train_hist.append(loss.item() / bs)
    leg_val_hist.append(val_mean)
    if val_mean <= best_val:
        best_val = val_mean
        best_legacy = copy.deepcopy(legacy)

    loss.backward()
    opt.step()
    leg_times.append(time.perf_counter() - t0)
    if e % 20 == 0 or e == EPOCHS - 1:
        print(f'[legacy:batched] epoch {e:4d}  train_bc={leg_train_hist[-1]:.4f}  '
              f'val_bc={val_mean:.4f}  best={best_val:.4f}  sec={leg_times[-1]:.2f}')

legacy = best_legacy
print(f'\nLegacy (batched) best val BC: {best_val:.6f}')
print(f'Total train time: {np.sum(leg_times):.0f}s '
      f'({np.mean(leg_times):.3f}s/epoch)')

In [ ]:
# Training curves side by side.
fig, (a0, a1) = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
a0.plot(result_new.train_loss, label='train loss', alpha=0.7)
a0.plot(result_new.val_loss, label='val BC', alpha=0.9)
a0.axvline(result_new.best_epoch, ls='--', c='k', lw=1, label=f'best @ {result_new.best_epoch}')
a0.set_title('New model (nonlinear SiLU)'); a0.set_xlabel('epoch'); a0.set_ylabel('loss / BC'); a0.legend()
a1.plot(leg_train_hist, label='train loss', alpha=0.7)
a1.plot(leg_val_hist, label='val BC', alpha=0.9)
a1.axvline(int(np.argmin(leg_val_hist)), ls='--', c='k', lw=1, label=f'best @ {int(np.argmin(leg_val_hist))}')
a1.set_title('Batched legacy (cNODE2)'); a1.set_xlabel('epoch'); a1.legend()
plt.tight_layout(); plt.show()

## 6. Predict + build the leave-one-out assemblages (shared)
`qtrn_*` = predicted intact composition for **every** community; `qtst_*` =
predicted composition for each leave-one-species-out assemblage. Both models
predict on the **same** `z` tensors, so the resulting keystoneness rows align
row-for-row.

Ground-truth `k_true` requires both real removal communities (`Ptest.csv`,
`Ztest.csv`, `Sample_id.csv`, `Species_id.csv`) **and** no read-depth filtering
(filtering reindexes taxa, desyncing the on-disk ids); otherwise we generate the
leave-one-out assemblages ourselves and report predicted keystoneness only.

In [ ]:
def build_loo(z_all):
    """Every leave-one-present-species-out assemblage from (n_samples, N).
    Returns (z_loo, sample_id, species_id) with 1-indexed ids."""
    z_cpu = z_all.detach().cpu()
    loo, sample_id, species_id = [], [], []
    for s in range(z_cpu.shape[0]):
        present = torch.nonzero(z_cpu[s] > 0).flatten().tolist()
        for sp in present:
            v = z_cpu[s].clone(); v[sp] = 0.0
            tot = float(v.sum())
            if tot <= 0:
                continue
            loo.append(v / tot)
            sample_id.append(s + 1); species_id.append(sp + 1)
    return torch.stack(loo), np.array(sample_id), np.array(species_id)


@torch.no_grad()
def legacy_predict(func, z, chunk=256):
    """Equilibrium for each row of z under the trained legacy model.
    Uses the 2-point output grid (same adaptive dopri5 solve as training,
    just fewer stored points) and chunks to bound memory."""
    func.eval()
    outs = []
    for i in range(0, z.shape[0], chunk):
        outs.append(integrate_set(func, z[i:i + chunk], T_PRED, batched=True))
    return torch.cat(outs, dim=0)


# Observed RELATIVE abundances from the filter-consistent loader tensors.
ptrn = data.p_all.detach().cpu().numpy().T            # (N, n_samples), cols sum to 1

_truth_files = ['Ztest.csv', 'Ptest.csv', 'Sample_id.csv', 'Species_id.csv']
_have_files = all(os.path.exists(os.path.join(DATA_DIR, f)) for f in _truth_files)
has_truth = _have_files and (MIN_READS <= 0)
if _have_files and MIN_READS > 0:
    print('Note: MIN_READS reindexes taxa/samples, so the on-disk ids are not used; '
          'generating leave-one-out assemblages from the filtered data instead.')

if has_truth:
    sample_id  = np.loadtxt(os.path.join(DATA_DIR, 'Sample_id.csv'),  delimiter=',').astype(int)
    species_id = np.loadtxt(os.path.join(DATA_DIR, 'Species_id.csv'), delimiter=',').astype(int)
    z_test_cpu = data.z_test.detach().cpu()
    ptst = data.p_test.detach().cpu().numpy().T       # (N, n_pairs), cols sum to 1
    z_full, z_loo = z_all_cpu, z_test_cpu
    print(f'Ground-truth removals present: {len(sample_id)} (sample, species) pairs '
          '-- computing predicted AND true keystoneness for both models.')
else:
    z_loo, sample_id, species_id = build_loo(z_all_cpu)
    ptst = np.zeros((ptrn.shape[0], z_loo.shape[0]), dtype=float)  # placeholder; pred ignores it
    z_full = z_all_cpu
    print(f'Computing predicted keystoneness only over {len(sample_id)} '
          'leave-one-out assemblages.')

# --- predictions: new model on its own device, legacy on CPU; both -> numpy ---
qtrn_new = predict(model, z_full.to(data.z_all.device)).detach().cpu().numpy()
qtst_new = predict(model, z_loo.to(data.z_all.device)).detach().cpu().numpy()
qtrn_leg = legacy_predict(legacy, z_full).numpy()
qtst_leg = legacy_predict(legacy, z_loo).numpy()

assert ptrn.max() <= 1.0 + 1e-6, 'ptrn not compositional -- expected columns summing to 1.'
print('predicted: qtrn_new', qtrn_new.shape, '| qtst_new', qtst_new.shape)

## 7. Reconstruction quality (does each model rebuild the communities?)
Per-sample Bray–Curtis between observed and predicted intact composition. Lower
is better; this is the precondition for trusting either model's keystoneness.

In [ ]:
def per_sample_bc(qtrn, ptrn):
    return (np.abs(ptrn.T - qtrn).sum(1)
            / np.clip(np.abs(ptrn.T + qtrn).sum(1), 1e-12, None))

bc_new = per_sample_bc(qtrn_new, ptrn)
bc_leg = per_sample_bc(qtrn_leg, ptrn)

fig, (a0, a1) = plt.subplots(1, 2, figsize=(12, 4.5))
a0.hist(bc_new, bins=30, alpha=0.6, label=f'new (mean {bc_new.mean():.3f})', color='#e34a33')
a0.hist(bc_leg, bins=30, alpha=0.6, label=f'legacy (mean {bc_leg.mean():.3f})', color='#2b8cbe')
a0.set_xlabel('per-sample Bray-Curtis (obs vs pred)'); a0.set_ylabel('# samples')
a0.set_title('Reconstruction error'); a0.legend()

a1.scatter(bc_leg, bc_new, s=8, alpha=0.4)
lim = [0, max(bc_leg.max(), bc_new.max())]
a1.plot(lim, lim, 'k--', lw=1)
a1.set_xlabel('legacy per-sample BC'); a1.set_ylabel('new per-sample BC')
a1.set_title('Below the line = new model reconstructs better')
plt.tight_layout(); plt.show()

print(f'mean reconstruction BC  ->  new: {bc_new.mean():.4f}   legacy: {bc_leg.mean():.4f}')

## 8. Structural keystoneness for both models
`Ks = BC(q_intact_renorm, q_removed) * (1 - p)`. The predicted score uses only
each model's own predictions (`qtrn_*`, `qtst_*`) and the observed abundance
`p`; `k_true` (when present) is identical across models since it is computed
from the observed removal communities, not from any model.

In [ ]:
ks_new = classical_structural_keystoneness(qtrn_new, qtst_new, ptrn, ptst, sample_id, species_id)
ks_leg = classical_structural_keystoneness(qtrn_leg, qtst_leg, ptrn, ptst, sample_id, species_id)

# Merge into one frame: shared keys + each model's predicted score (+ shared truth).
ks = ks_new[['sample', 'species', 'p_species']].copy()
ks['k_new'] = ks_new['k_pred'].to_numpy()
ks['k_legacy'] = ks_leg['k_pred'].to_numpy()
if has_truth:
    ks['k_true'] = ks_new['k_true'].to_numpy()   # same as ks_leg['k_true']

os.makedirs('/content/results', exist_ok=True)
ks.to_csv('/content/results/keystoneness_new_vs_legacy.csv', index=False)
print('wrote /content/results/keystoneness_new_vs_legacy.csv  rows =', len(ks))
ks.sort_values('k_new', ascending=False).head(10)

In [ ]:
# Do the two models agree on which (sample, species) pairs are keystones?
rho = spearman(ks['k_new'], ks['k_legacy'])
pear = np.corrcoef(ks['k_new'], ks['k_legacy'])[0, 1]

plt.figure(figsize=(5.5, 5.5))
plt.scatter(ks['k_legacy'], ks['k_new'], s=8, alpha=0.3)
lim = [0, max(ks['k_legacy'].max(), ks['k_new'].max())]
plt.plot(lim, lim, 'k--', lw=1)
plt.xlabel('legacy $k_{pred}$'); plt.ylabel('new $k_{pred}$')
plt.title(f'Per-pair keystoneness agreement\nSpearman rho = {rho:.3f}  |  Pearson r = {pear:.3f}')
plt.tight_layout(); plt.show()

## 9. Which model tracks the truth better? (needs ground-truth removals)
When real post-removal communities exist, `k_true` is the target. We score each
model's `k_pred` against it (Spearman rho + Pearson r). The higher-correlating
model is the one whose learned assembly rules better reproduce the *measured*
keystone effect.

In [ ]:
if has_truth:
    fig, axes = plt.subplots(1, 2, figsize=(11, 5), sharex=True, sharey=True)
    for ax, col, name, c in [(axes[0], 'k_new', 'new (nonlinear)', '#e34a33'),
                             (axes[1], 'k_legacy', 'batched legacy', '#2b8cbe')]:
        rho = spearman(ks['k_true'], ks[col])
        pear = np.corrcoef(ks['k_true'], ks[col])[0, 1]
        ax.scatter(ks['k_true'], ks[col], s=8, alpha=0.3, color=c)
        lim = [0, max(ks['k_true'].max(), ks[col].max())]
        ax.plot(lim, lim, 'k--', lw=1)
        ax.set_xlabel('$k_{true}$'); ax.set_ylabel('$k_{pred}$')
        ax.set_title(f'{name}\nSpearman rho = {rho:.3f}  |  r = {pear:.3f}')
    plt.tight_layout(); plt.show()
else:
    print('No ground-truth removals -- skipping the k_pred vs k_true comparison. '
          'Use the bundled set (USE_BUNDLED=True, MIN_READS=0) to enable it.')

## 10. Top keystone species: do the rankings agree?
The paper ranks taxa by **median keystoneness across communities**. We compare
the two models' top-20 by median `k_pred`.

In [ ]:
med_new = ks.groupby('species')['k_new'].median().sort_values(ascending=False)
med_leg = ks.groupby('species')['k_legacy'].median().sort_values(ascending=False)

top_new = list(med_new.head(20).index)
top_leg = list(med_leg.head(20).index)
overlap = sorted(set(top_new) & set(top_leg))
print(f'Top-20 species overlap: {len(overlap)}/20  -> {overlap}')

# Side-by-side bars for whichever species land in either top-20.
species_union = sorted(set(top_new) | set(top_leg))
x = np.arange(len(species_union))
vals_new = med_new.reindex(species_union).fillna(0).to_numpy()
vals_leg = med_leg.reindex(species_union).fillna(0).to_numpy()
plt.figure(figsize=(12, 4.5))
plt.bar(x - 0.2, vals_new, width=0.4, label='new (nonlinear)', color='#e34a33')
plt.bar(x + 0.2, vals_leg, width=0.4, label='batched legacy', color='#2b8cbe')
plt.xticks(x, [str(s) for s in species_union], rotation=60)
plt.xlabel('species (1-indexed)'); plt.ylabel('median $k_{pred}$')
plt.title('Median structural keystoneness per species (top-20 union)'); plt.legend()
plt.tight_layout(); plt.show()

## 11. Summary

In [ ]:
summary = {
    'model': ['new (nonlinear SiLU)', 'batched legacy (cNODE2)'],
    'best_val_bc': [result_new.best_val_loss, best_val],
    'mean_recon_bc': [bc_new.mean(), bc_leg.mean()],
    'total_train_s': [float(np.sum(result_new.epoch_seconds)), float(np.sum(leg_times))],
    'mean_epoch_s': [float(np.mean(result_new.epoch_seconds)), float(np.mean(leg_times))],
}
if has_truth:
    summary['spearman_vs_true'] = [
        spearman(ks['k_true'], ks['k_new']),
        spearman(ks['k_true'], ks['k_legacy']),
    ]
summary_df = pd.DataFrame(summary)
summary_df.to_csv('/content/results/new_vs_legacy_summary.csv', index=False)
print('Spearman(new k_pred, legacy k_pred) =',
      round(spearman(ks['k_new'], ks['k_legacy']), 4))
summary_df

## 12. Download results

In [ ]:
!ls -la /content/results
try:
    from google.colab import files
    files.download('/content/results/keystoneness_new_vs_legacy.csv')
    files.download('/content/results/new_vs_legacy_summary.csv')
except Exception as e:
    print('Not in Colab or download skipped:', e)